# Dataset Setup — CrisisMMD

This notebook performs initial setup and unification of the CrisisMMD annotation files.

It reads multiple `.tsv` files from the dataset directory, fixes encoding and text anomalies, and consolidates all rows into a single `all_annotations.tsv` file to be used in downstream tasks.

## Imports and setup

We use a `.env` file to store the paths to the `annotations` directory and the main `data` directory. This allows us to separate configuration from code and ensure portability.


In [ ]:
import pandas as pd
import os
from ftfy import fix_text

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
# Load the "annotations" directory path from .env file
annotations_dir = os.getenv("ANNOTATIONS_DIR")
data_dir = os.getenv("DATA_DIR")


## Inspect raw annotation files

We list all the `.tsv` files in the annotations directory. Each file corresponds to annotations from a different disaster event or Twitter stream.


In [ ]:
# List all .tsv files in the folder
files = os.listdir(annotations_dir)
for file in files:
	print(file)

## Merge and normalize annotations

We iterate over each `.tsv` file to:
- Append a `source_file` column to track provenance
- Map specific disasters to a new general `disaster_type` column
- Fix broken or inconsistent Unicode using `ftfy`
- Concatenate everything into a single unified DataFrame

This process ensures that we retain full traceability while producing an easily accessible input dataset.


In [ ]:
def map_to_general_disaster_type(specific_type):
	"""Map specific disaster types to more general categories"""
	if 'hurricane' in specific_type:
		return 'hurricane'
	elif 'wildfire' in specific_type:
		return 'wildfire'
	elif 'earthquake' in specific_type:
		return 'earthquake'
	elif 'flood' in specific_type:
		return 'flood'
	else:
		return 'other'

In [ ]:
# Get all .tsv files in the "annotations" directory
tsv_files = [f for f in os.listdir(annotations_dir) if f.endswith(".tsv")]

# List to collect DataFrames for each disaster type
df_list = []

# Iterate over each .tsv file and read its data
print("Files read:")
for filename in tsv_files:
	# Add disaster type based on file name
	disaster_type = filename.replace('_final_data.tsv', '')

	file_path = os.path.join(annotations_dir, filename)
	df = pd.read_csv(file_path, sep="\t")
	print(f"{filename}: {len(df)} rows")

	# Add a column indicating the disaster type
	df["disaster_type"] = disaster_type

	# Add general disaster type
	df['general_disaster_type'] = df['disaster_type'].apply(map_to_general_disaster_type)

	df_list.append(df)

# Merge all DataFrames into a single DataFrame
annotations = pd.concat(df_list, ignore_index=True)
print(f"\nMerged file size: {len(annotations)} rows")

In [ ]:
annotations['tweet_text'] = annotations['tweet_text'].apply(fix_text)

In [ ]:
pd.set_option('display.max_colwidth', None)
annotations['tweet_text'].head(5)

## Save unified annotations file

We export the merged dataset to `annotations.tsv`, which contains the full set of preprocessed rows from all input files.

Basic statistics like number of rows and unique tweet IDs are printed for verification.


In [ ]:
save_path = os.path.join(data_dir, "annotations.tsv")

# Save the merged DataFrame as a .tsv file
annotations.to_csv(save_path, sep='\t', index=False)

In [ ]:
annotations.head(2)

In [ ]:
# Check if all the disaster types were read correctly
disasters_type = annotations["disaster_type"].unique()

print(disasters_type)
print(f"\n Total Natural Disasters: {str(len(disasters_type))}")